# 05 - Configure software settings and understand units

These examples work offline. They configure Python timeouts and logging, and preview
angle conversion. The library also provides `query_settings()` and verified `configure()`
for supported velocity, jog, backlash and button groups. Homing parameters remain read-only;
unverified mappings remain unavailable. This notebook sends no device-setting, EEPROM or
firmware writes. See the GUI contract for those separately authorized operations.

Run cells from top to bottom in a fresh Python kernel. This notebook stands alone.
Install and launch as described in [the tutorial index](README.md).
Hardware access is disabled by default. Set your own eight-digit serial; no unit is
selected automatically. Close other notebooks' connections and other control software
before accessing the device. One connection owns the USB receive path.


## Timeout settings

Use a dummy eight-digit serial below only to construct an offline object. Construction
does not access USB. Query timeout bounds individual requests; motion timeout bounds a
whole home/move; cleanup timeout gives a separate budget to a stop after a motion failure.
All values must be finite and positive. USB/Windows calls cannot offer hard real-time bounds.


In [ ]:
from k10cr2 import K10CR2

settings = {"query_timeout": 3.0, "motion_timeout": 120.0, "cleanup_timeout": 5.0}
preview = K10CR2("00000000", **settings)  # Offline only: do not connect this serial.
print({name: getattr(preview, name) for name in settings})

Each query/motion/stop also accepts a per-call `timeout=`. A query within a motion is
still capped by `query_timeout`. For a live connection, use the settings dictionary in
`K10CR2(SERIAL, **settings)`. A larger timeout does not fix a wrong driver or ownership conflict.


## Logging

Enable raw APT traffic on the transport logger when diagnosing a live session. Restore the
previous logger level when done. Library imports and logging setup alone do not open USB.


In [ ]:
import logging

logging.basicConfig(level=logging.INFO)
transport_logger = logging.getLogger("k10cr2.transport")
previous_level = transport_logger.level
transport_logger.setLevel(logging.DEBUG)
print("Raw APT traffic logging enabled for future explicit connections.")

In [ ]:
transport_logger.setLevel(previous_level)
print("Previous transport logging level restored.")

## Preview angle quantization

The scale is `200 * 2048 * 120 = 49,152,000` counts per revolution. Use the calculation below for the degree/radian increments;
these increments describe command resolution, not angular accuracy.

Fixed-range targets require finite values in `0 <= angle <= 360`. The 360-degree
endpoint is one complete counted turn. Conversion uses nearest-count rounding with ties to even. It never wraps.


In [ ]:
import math

from k10cr2.protocol import COUNTS_PER_REVOLUTION
from k10cr2.settings import resolve_target

print("Counts per revolution:", COUNTS_PER_REVOLUTION)
print("Degrees per count:", 360 / COUNTS_PER_REVOLUTION)
print("Radians per count:", math.tau / COUNTS_PER_REVOLUTION)
for target in (0.0, 1.0, 45.0, 90.0, 359.0):
    counts = resolve_target(target, 0)
    actual = counts * 360 / COUNTS_PER_REVOLUTION
    print(
        {
            "requested_deg": target,
            "counts": counts,
            "quantized_deg": actual,
            "rounding_error_deg": actual - target,
        }
    )

In [ ]:
for invalid in (-1, 360.1, float("nan"), float("inf"), True):
    try:
        resolve_target(invalid, 0)
    except ValueError as error:
        print(repr(invalid), "->", str(error))

The target resolver is in `k10cr2.settings`; normal application code can pass degrees
directly to `move_absolute()`. See [the units contract](../../docs/commands.md#units-and-status)
for the scale and source references.

Next: [06 - Homing](06_homing.ipynb).
